# Neural Style Transfer - A Complete Guide

---

**What is this notebook about?**

This notebook teaches you how to perform **Neural Style Transfer** - a fascinating technique that combines:
- The **content** of one image (e.g., a photo of a face)
- The **style** of another image (e.g., Van Gogh's Starry Night painting)

The result is a new image that looks like your photo painted in the style of the artwork!

---

**A Brief History:**

Neural Style Transfer was introduced in 2015 by Leon Gatys, Alexander Ecker, and Matthias Bethge in their paper "A Neural Algorithm of Artistic Style". It was one of the first demonstrations that deep neural networks had learned meaningful representations of visual art and style.

---

**How does it work? (The Big Picture)**

1. We start with a random noise image (or copy of the content image)
2. We use a pre-trained CNN (VGG16) to extract "features" from images
3. We optimize our image so that:
   - Its **high-level features** match the content image (preserves structure)
   - Its **texture patterns** match the style image (adds artistic style)

The key insight is that different layers of a CNN capture different types of information:
- **Early layers**: Simple patterns like edges, colors, textures
- **Later layers**: Complex patterns like shapes, objects, faces

---

In [ ]:
# ============================================================================
# INTERACTIVE VISUALIZATIONS -- setup (run this cell ONCE).
# Each "🎮 Interactive" cell below loads a standalone HTML file from the
# published copy on GitHub Pages, in an isolated <iframe> (its CSS/JS can't leak
# into the notebook). It is shown FULL WIDTH and auto-fits its content height.
# No local files needed -- the visualizations are read from GitHub.
# ============================================================================
from IPython.display import HTML

def show_viz(path, height="600px"):
    """Embed an interactive visualization full width; it auto-fits its height.
    `path` may be a bare 'interactive_viz/<file>.html' (resolved to the GitHub
    Pages copy) or a full https URL."""
    base = "https://shammun.github.io/shammunul-fastai-notes/notebooks/"
    if not path.startswith("http"):
        path = base + path
    return HTML(
        f'<iframe src="{path}" loading="lazy" allowfullscreen '
        f'style="width:100%;height:{height};border:1px solid #dde5f2;border-radius:12px;'
        f'box-shadow:0 8px 24px rgba(123,92,214,.12);background:#fff;"></iframe>'
        '<script>addEventListener("message",function(e){'
        'if(e.data&&e.data.type==="ce-frame-height"&&e.data.height>50){'
        'var fs=document.querySelectorAll("iframe");for(var i=0;i<fs.length;i++){'
        'if(fs[i].contentWindow===e.source){fs[i].style.height=e.data.height+"px";break;}}}});</script>'
    )

### 🎮 Before any code: watch the whole algorithm once

Style transfer has an unusual shape: **nothing in the network trains — the image itself does.** The interactive below walks the full loop in six clickable stages (the three images → frozen VGG16 → content loss → style loss → one combined number → gradients landing on the *pixels*). Press **▶ Play** to loop whole optimization iterations and watch the noise thumbnail turn into a stylized image. Every box in the diagram is clickable, and the code panel underneath always shows the exact notebook code for the lit-up stage — all of which you'll build piece by piece below.

In [ ]:
# ============================================================================
# 🎮 INTERACTIVE -- run this cell. It loads ./interactive_viz/nst_pipeline_overview.html
# at full width, sized to fit the visualization (setup cell near the top required).
# The complete NST pipeline as 6 clickable stages with synced notebook code.
# ▶ Play loops optimization iterations -- watch model.t evolve from noise to art.
# Click any box in the diagram (or the stage chips) to jump to that stage.
# ============================================================================
show_viz("interactive_viz/nst_pipeline_overview.html", height="700px")

## Section 1: Setup and Imports

Let's import all the libraries we'll need.

In [ ]:
# ============================================================================
# STANDARD PYTHON LIBRARIES
# ============================================================================

import pickle, gzip, math, os, time, shutil, torch, random, timm, torchvision, io, PIL
# - pickle, gzip: For saving/loading compressed data
# - math: Mathematical functions
# - os, shutil: File system operations
# - time: Timing operations
# - torch: PyTorch - the main deep learning library
# - random: Random number generation
# - timm: PyTorch Image Models - library with pre-trained models
# - torchvision: Computer vision utilities for PyTorch
# - io, PIL: Image input/output handling

import fastcore.all as fc  # Utility library with helpful Python extensions
import matplotlib as mpl   # Plotting library
import numpy as np         # Numerical computing
import matplotlib.pyplot as plt  # Plotting interface

from collections.abc import Mapping  # Type checking for dict-like objects
from pathlib import Path             # Modern file path handling
from operator import attrgetter, itemgetter  # Functions to get attributes/items
from functools import partial        # Create partial functions
from copy import copy                # Object copying
from contextlib import contextmanager  # Context managers

# ============================================================================
# PYTORCH AND TORCHVISION IMPORTS
# ============================================================================

import torchvision.transforms.functional as TF  # Image transformations
import torch.nn.functional as F                  # Neural network functions
from torchvision import transforms               # More image transforms

from torch import tensor, nn, optim
# - tensor: Create PyTorch tensors
# - nn: Neural network modules
# - optim: Optimization algorithms

from torch.utils.data import DataLoader, default_collate  # Data loading utilities
from torch.nn import init                                  # Weight initialization
from torch.optim import lr_scheduler                       # Learning rate scheduling
from torcheval.metrics import MulticlassAccuracy          # Metrics
from datasets import load_dataset, load_dataset_builder    # HuggingFace datasets
from fastcore.foundation import L, store_attr              # fastcore utilities

# ============================================================================
# MINIAI IMPORTS (Custom training library from previous notebooks)
# ============================================================================

from miniai.datasets import *    # Dataset utilities
from miniai.conv import *        # Convolutional layer utilities
from miniai.learner import *     # The Learner class for training
from miniai.activations import * # Activation functions
from miniai.init import *        # Initialization utilities
from miniai.sgd import *         # Optimization utilities
from miniai.resnet import *      # ResNet utilities

**What does the code above do?**

This imports all the libraries we need:

- **timm**: "PyTorch Image Models" - a library containing many pre-trained computer vision models. We'll use VGG16 from here.
- **torchvision**: PyTorch's computer vision library with image loading, transforms, and models.
- **fastcore**: A utility library that adds helpful features to Python (like the enhanced `L` list class).
- **miniai**: Our custom training library built in earlier notebooks.

In [ ]:
# URLs for the images we'll use in our style transfer demo
# You can change these to any images you like!

# Content image: A face photo from Pexels (free stock photos)
# This will be the "structure" we want to keep
face_url = "https://images.pexels.com/photos/2690323/pexels-photo-2690323.jpeg?w=256"

# Style image: A spiderweb with water droplets
# The texture/pattern from this image will be applied to the face
spiderweb_url = "https://images.pexels.com/photos/34225/spider-web-with-water-beads-network-dewdrop.jpg?w=256"

**What does the code above do?**

We define URLs for two images:
1. **Content image (face_url)**: The image whose structure/content we want to preserve
2. **Style image (spiderweb_url)**: The image whose artistic style/texture we want to transfer

The `?w=256` at the end tells Pexels to give us a 256-pixel wide version (smaller = faster processing).

**Feel free to experiment!** Try different images:
- Famous paintings (Van Gogh, Picasso, Monet)
- Textured patterns (brick walls, fabric, water)
- Your own photos

---

## Section 2: Loading Images

First, we need a function to download images from URLs and convert them to PyTorch tensors.

In [ ]:
def download_image(url):
    """
    Download an image from a URL and convert it to a PyTorch tensor.
    
    Parameters:
    -----------
    url : str
        The URL of the image to download
        
    Returns:
    --------
    tensor : torch.Tensor
        Image as a tensor with shape (3, H, W) and values in [0, 1]
        - 3 channels: Red, Green, Blue
        - H: Height in pixels
        - W: Width in pixels
    """
    # Step 1: Download the raw image bytes from the URL
    # fc.urlread fetches content from a URL
    # decode=False returns raw bytes instead of trying to decode as text
    imgb = fc.urlread(url, decode=False)
    
    # Step 2: Convert bytes to a tensor and decode the image
    # tensor(list(imgb), dtype=torch.uint8) converts bytes to a tensor of integers 0-255
    # torchvision.io.decode_image decodes JPEG/PNG bytes into a tensor
    # Result shape: (3, H, W) with values 0-255
    img_tensor = torchvision.io.decode_image(tensor(list(imgb), dtype=torch.uint8))
    
    # Step 3: Convert to float and normalize to [0, 1] range
    # .float() converts from int (0-255) to float
    # /255. scales values from [0, 255] to [0, 1]
    return img_tensor.float() / 255.

**What does the code above do?**

The `download_image` function:

1. **Downloads** the image from the URL as raw bytes
2. **Decodes** the image (JPEG/PNG) into a tensor of pixel values
3. **Normalizes** pixel values from 0-255 to 0.0-1.0

**Why normalize to [0, 1]?**

Neural networks work better with normalized inputs:
- Original pixels: integers from 0 to 255
- Normalized: floats from 0.0 to 1.0
- This keeps values in a reasonable range for gradient-based optimization

In [ ]:
# Download the content image and move it to the GPU (if available)
# def_device is defined in miniai - it's 'cuda' if GPU is available, else 'cpu'
content_im = download_image(face_url).to(def_device)

# Print the shape to verify:
# Should be (3, H, W) where 3 = RGB channels
print('content_im.shape:', content_im.shape)

# Display the image
show_image(content_im);

**What does the code above do?**

1. **`download_image(face_url)`**: Downloads and processes the image
2. **`.to(def_device)`**: Moves the tensor to GPU if available (much faster!)
3. **`print('content_im.shape:',...)`: Shows the tensor dimensions
4. **`show_image(content_im)`**: Displays the image

**Expected shape:** `(3, 256, 256)` - 3 color channels, 256x256 pixels

In [ ]:
# Verify that pixel values are in the expected [0, 1] range
# min() should be close to 0 (darkest pixels)
# max() should be close to 1 (brightest pixels)
content_im.min(), content_im.max()

**What does the code above do?**

Checks that our image values are properly normalized:
- `min()` should be around 0 (representing black/dark pixels)
- `max()` should be around 1 (representing white/bright pixels)

If values were outside [0, 1], it could cause problems during optimization.

---

## Section 3: Optimizing Images

Here's a key insight: **We can treat an image itself as the thing we're optimizing!**

Normally in deep learning:
- We have a fixed input (image)
- We optimize the model's weights

In style transfer:
- We have a fixed model (VGG16, pre-trained)
- We optimize the image pixels!

This is called **"image optimization"** or **"inverting the network"**.

### Creating a Dummy DataLoader

Our training framework (miniai's Learner) expects a DataLoader. Since we're optimizing a single image (not iterating through a dataset), we create a "dummy" DataLoader that just counts iterations.

In [ ]:
class LengthDataset():
    """
    A dummy dataset that just returns zeros.
    Used to control how many optimization steps we take.
    
    Parameters:
    -----------
    length : int
        Number of "samples" (actually just iteration count)
    """
    def __init__(self, length=1): 
        self.length = length  # Store the length
    
    def __len__(self): 
        return self.length    # Return length when len() is called
    
    def __getitem__(self, idx): 
        return 0, 0           # Return dummy data (we don't use it)


def get_dummy_dls(length=100):
    """
    Create dummy DataLoaders for image optimization.
    
    Parameters:
    -----------
    length : int
        Number of optimization steps to take during "training"
        
    Returns:
    --------
    DataLoaders
        A DataLoaders object with train and valid loaders
    """
    return DataLoaders(
        DataLoader(LengthDataset(length), batch_size=1),  # Train: 'length' steps
        DataLoader(LengthDataset(1), batch_size=1)        # Valid: 1 step (not really used)
    )

**What does the code above do?**

Creates fake/dummy data loaders because:
- Our Learner framework expects DataLoaders
- But we're not actually loading data - we're optimizing a single image
- The `length` parameter controls how many optimization steps we take

**Example:**
```python
get_dummy_dls(100)  # Creates loaders that will iterate 100 times
```

This is a clever trick to reuse our training framework for a different purpose!

### The TensorModel: Making an Image Optimizable

In PyTorch, to optimize something with gradient descent, it needs to be a `nn.Parameter`. We create a simple "model" where the only parameter is the image tensor itself.

In [ ]:
class TensorModel(nn.Module):
    """
    A "model" that just holds a tensor as a learnable parameter.
    This lets us optimize the tensor's values using gradient descent.
    
    Parameters:
    -----------
    t : torch.Tensor
        The tensor to optimize (e.g., an image)
    """
    def __init__(self, t):
        super().__init__()  # Initialize the parent nn.Module class
        
        # nn.Parameter wraps a tensor and tells PyTorch:
        # "This should be optimized during training"
        # t.clone() creates a copy so we don't modify the original
        self.t = nn.Parameter(t.clone())
    
    def forward(self, x=0):
        """
        Forward pass - just returns the tensor.
        
        The x parameter is ignored (it's there for compatibility
        with the training loop which passes batch data).
        """
        return self.t

**What does the code above do?**

Creates a PyTorch "model" where the image IS the model:

1. **`nn.Parameter(t.clone())`**: Wraps the image tensor so PyTorch knows to:
   - Track gradients for it
   - Update it during optimization

2. **`forward(self, x=0)`**: Returns the image tensor. The `x` parameter is just for compatibility with the training loop.

**The key insight:**
```
Normal training:   Fixed input → Trainable model → Output → Loss
Image optimization: Trainable image → Fixed model → Output → Loss
```

In [ ]:
# Create a TensorModel with a random image (same size as our content image)
# torch.rand_like creates a tensor of random values [0, 1] with the same shape
model = TensorModel(torch.rand_like(content_im))

# Display the random starting image
# model() calls the forward method, returning the image tensor
show_image(model());

**What does the code above do?**

1. **`torch.rand_like(content_im)`**: Creates a tensor of random values between 0 and 1, with the same shape as `content_im`
2. **`TensorModel(...)`**: Wraps it in our model class
3. **`model()`**: Gets the current image (random noise at this point)
4. **`show_image(...)`**: Displays it

You should see random colored noise - this is our starting point!

In [ ]:
# Check what parameters the model has
# Should be just one parameter: the image tensor
[p.shape for p in model.parameters()]

[torch.Size([3, 256, 256])]

**What does the code above do?**

Lists all trainable parameters in the model:
- `model.parameters()` returns an iterator over all `nn.Parameter` objects
- We just have one: the image with shape `[3, 256, 256]`

This confirms that our "model" is just a single trainable image!

### Custom Callback for Image Optimization

We need to customize how our training loop works for image optimization.

In [ ]:
class ImageOptCB(TrainCB):
    """
    Callback that adapts the training loop for image optimization.
    
    Key differences from normal training:
    1. predict: The model output IS the image (no input needed)
    2. get_loss: Loss function only takes the predicted image (no target from batch)
    """
    
    def predict(self, learn):
        """
        Get the current image from the model.
        
        In normal training: preds = model(input)
        Here: preds = model() (no input needed, model IS the image)
        """
        learn.preds = learn.model()  # Just call the model with no arguments
    
    def get_loss(self, learn):
        """
        Calculate the loss.
        
        In normal training: loss = loss_func(preds, targets)
        Here: loss = loss_func(preds) (target is built into the loss function)
        """
        learn.loss = learn.loss_func(learn.preds)  # Loss function handles the target internally

**What does the code above do?**

Creates a callback that modifies how training works:

| Normal Training | Image Optimization |
|-----------------|--------------------|
| `preds = model(batch_input)` | `preds = model()` (image is the model) |
| `loss = loss_fn(preds, batch_target)` | `loss = loss_fn(preds)` (target is fixed) |

**Why this design?**

Our loss function will compare the current image against fixed target images (content and style). The targets are stored in the loss function itself, not passed from a data batch.

### Demo: Simple MSE Loss Optimization

Let's first test our setup with a simple loss: Mean Squared Error (MSE) between our optimizable image and the content image. This will try to recreate the content image pixel-by-pixel.

In [ ]:
def loss_fn_mse(im):
    """
    Simple MSE loss comparing the current image to the content image.
    
    Parameters:
    -----------
    im : torch.Tensor
        The current state of our optimizable image
        
    Returns:
    --------
    loss : torch.Tensor
        The mean squared error between im and content_im
    """
    # F.mse_loss computes: mean((im - content_im)^2)
    # This measures how different the images are pixel-by-pixel
    return F.mse_loss(im, content_im)


### What Jeremy Says About This Section

This serves as a **warm-up exercise** — the MSE loss is intentionally simple: it just measures the pixel-by-pixel difference between the generated image (starting from random noise) and the target content image.

Jeremy describes it as a *"very simple loss"* where there is essentially only *"one direction that you update"* — which makes it *"almost trivial to solve."* The point isn't the task itself, but rather getting the **training infrastructure** in place (the `Learner`, callbacks, dummy dataloader, optimizable `TensorModel`) before tackling something harder.

The goal is to demonstrate a key idea: **noisy pixels can be transformed into something meaningful simply by following a loss function** — in this case, just making the pixels look as close as possible to the target photo.

After 100 optimization steps with Adam, the loss drops close to zero and the generated image becomes practically identical to the content image. ✓

In [ ]:
# Create a new model starting from random noise
model = TensorModel(torch.rand_like(content_im))

# Set up callbacks:
# - ImageOptCB: Our custom callback for image optimization
# - ProgressCB: Shows a progress bar
# - MetricsCB: Tracks and displays metrics (like loss)
# - DeviceCB: Ensures data is on the right device (CPU/GPU)
cbs = [ImageOptCB(), ProgressCB(), MetricsCB(), DeviceCB()]

# Create the Learner:
# - model: Our TensorModel (the optimizable image)
# - get_dummy_dls(100): Fake dataloader for 100 optimization steps
# - loss_fn_mse: Our MSE loss function
# - lr=1e-2: Learning rate of 0.01
# - opt_func=torch.optim.Adam: Use the Adam optimizer
learn = Learner(model, get_dummy_dls(100), loss_fn_mse, 
                lr=1e-2, cbs=cbs, opt_func=torch.optim.Adam)

# Run 1 "epoch" (which is really 100 optimization steps)
learn.fit(1)

**What does the code above do?**

1. **`loss_fn_mse(im)`**: Computes the mean squared error between our current image and the target content image. Lower = more similar.

2. **`TensorModel(torch.rand_like(content_im))`**: Creates a model holding random noise.

3. **Callbacks setup**:
   - `ImageOptCB()`: Our custom callback for image optimization
   - `ProgressCB()`: Shows progress bar
   - `MetricsCB()`: Tracks metrics
   - `DeviceCB()`: Handles GPU/CPU

4. **`Learner(...)`**: Combines everything into a training pipeline.

5. **`learn.fit(1)`**: Runs 100 optimization steps (1 "epoch" of our dummy dataloader).

**What should happen:**
The random noise should gradually transform to look like the content image!

## The Key Insight: `loss_fn_mse` is a closure

```python
content_im = ...  # defined in outer scope

def loss_fn_mse(im):
    return F.mse_loss(im, content_im)  # content_im is "baked in"
```

`loss_fn_mse` **doesn't need a target passed in** because it already *captured* `content_im` from the surrounding scope when it was defined. This is a Python closure — the target is hardcoded into the function itself.

---

## How they're glued together

**Step 1 — You pass `loss_fn_mse` into `Learner`:**
```python
learn = Learner(model, get_dummy_dls(100), loss_fn_mse, ...)
#                                          ^^^^^^^^^^^^
#                               stored as learn.loss_func
```
So `learn.loss_func` **is** `loss_fn_mse`.

---

**Step 2 — `ImageOptCB.predict` runs first:**
```python
def predict(self, learn): learn.preds = learn.model()
```
`learn.model()` calls `TensorModel.forward()`, which just returns the raw optimizable image tensor. That tensor gets stored as `learn.preds`.

---

**Step 3 — `ImageOptCB.get_loss` runs next:**
```python
def get_loss(self, learn): learn.loss = learn.loss_func(learn.preds)
```
Expanding this:
```python
learn.loss = loss_fn_mse(learn.preds)
#                        ^^^^^^^^^^^ the generated image
# inside loss_fn_mse:
#   F.mse_loss(im, content_im)
#              ^^  ^^^^^^^^^^
#         preds    baked-in target
```
No explicit target needed — `content_im` was already captured inside `loss_fn_mse`.

---

## The full `learn.fit(1)` loop

```
for each step (100 steps):
    │
    ├─ predict()  →  learn.preds = model()             # get current image
    │
    ├─ get_loss() →  learn.loss  = loss_fn_mse(preds)  # compare to content_im
    │
    ├─ backward() →  compute gradients w.r.t. pixel values
    │
    └─ step()     →  Adam updates the pixel values to reduce loss
```

The `Learner` itself never sees the target directly. The target is hidden *inside* `loss_fn_mse` as a closure. This is why `get_loss` only needs predictions — the comparison is already encoded in the loss function.

---

## Why design it this way?

In normal supervised learning, `loss_func(preds, targets)` takes two arguments. But here **there are no real batches** — the dummy dataloader just ticks 100 times. The "target" (`content_im`) never changes, so it's more natural to bake it into the loss function rather than pass it through the dataloader machinery.

This pattern will become more powerful in style transfer, where the loss function captures *both* style features and content features from pre-computed VGG activations — again, all baked in as closures.

# Why Does `learn.fit(1)` Run 100 Optimization Steps?

## The Confusion

`fit(1)` looks like it should run **once**. But it actually runs **100 steps**. Where does "100" come from?

**Short answer:** `fit(1)` means "1 epoch," and one epoch = one full pass through the DataLoader. The DataLoader has 100 batches in it (because of `get_dummy_dls(100)`), so one epoch = 100 steps.

---

## Step-by-Step Trace

### Step 1 — The dummy dataset pretends to have 100 "samples"

```python
class LengthDataset():
    def __init__(self, length=1): 
        self.length = length
    def __len__(self): 
        return self.length        # ← tells Python: "I have 100 items"
    def __getitem__(self, idx): 
        return 0, 0               # ← returns garbage (we never use it)
```

This dataset doesn't hold real data. Its **only job** is to answer *"how many items do you have?"* with whatever number you give it.

---

### Step 2 — The DataLoader wraps this dataset with `batch_size=1`

```python
def get_dummy_dls(length=100):
    return DataLoaders(
        DataLoader(LengthDataset(length), batch_size=1),  # train loader
        ...
    )
```

A `DataLoader` iterates through a dataset in batches. With **100 items** and **batch_size=1**, the DataLoader will yield **100 batches**. Each batch is just `(0, 0)` — throwaway values.

---

### Step 3 — You pass this into the Learner

```python
learn = Learner(model, get_dummy_dls(100), loss_fn_mse, ...)
```

Now the Learner's training DataLoader has **100 batches** in it.

---

### Step 4 — `learn.fit(1)` means "run 1 epoch"

In any training framework, one **epoch** = one complete pass through the DataLoader. So what `fit(1)` does internally is roughly:

```python
def fit(self, n_epochs):
    for epoch in range(n_epochs):        # n_epochs = 1, so just once
        for batch in self.dls.train:     # ← THIS loops 100 times!
            preds = self.model()
            loss = self.loss_func(preds)
            loss.backward()
            self.opt.step()
```

The outer loop runs **once** (1 epoch). The **inner loop** runs **100 times** because `self.dls.train` yields 100 batches.

---

## The Full Chain of Causation

```
get_dummy_dls(100)
    → LengthDataset(100)
        → __len__() returns 100
            → DataLoader produces 100 batches
                → fit(1) loops over those 100 batches
                    → 100 optimization steps
```

---

## The Key Insight

The **"100" doesn't come from `fit(1)`** — it comes from **`get_dummy_dls(100)`**.

- Change to `get_dummy_dls(300)` → `fit(1)` runs **300** steps.
- Call `fit(3)` with `get_dummy_dls(100)` → **3 × 100 = 300** steps.

This is a clever hack: the `Learner` framework was built for normal training where you iterate over real data batches. Here there's no real data — you just want to run the optimize-the-pixels loop N times. So you **trick the framework** by giving it a fake dataset of length N, and the existing machinery does the rest.

### 🎮 Interactive: pixels ARE the parameters

Everything above (`TensorModel`, `ImageOptCB`, the closure, the 100 steps) exists to make one inversion work: **the optimizer's parameter tensor is the image.** The lab below runs that exact MSE demo on a 9×9 image so you can see every moving part: forward just *returns the pixels*, the loss compares them to the target, `backward()` puts a gradient on *every pixel* (red = too bright, blue = too dark), and `step()` nudges them. **Click anywhere on the loss curve to jump to that iteration**, and drag the lr slider to feel why the curve is an exponential decay.

In [ ]:
# ============================================================================
# 🎮 INTERACTIVE -- run this cell. It loads ./interactive_viz/pixels_as_parameters.html
# at full width, sized to fit the visualization (setup cell near the top required).
# A 9x9 noise image gradient-descends toward a target '7' under plain MSE --
# the notebook's loss_fn_mse demo in slow motion (4 stages per iteration).
# CLICK the loss curve to jump to any iteration; lr + speed sliders included.
# ============================================================================
show_viz("interactive_viz/pixels_as_parameters.html", height="740px")

In [ ]:
# Compare the result (left) with the target (right)
# .clip(0, 1) ensures values stay in valid range for display
show_images([learn.model().clip(0, 1), content_im]);

![](img_14.png)

**What does the code above do?**

Shows the result of our optimization:
- **Left**: Our optimized image
- **Right**: The target content image

**`.clip(0, 1)`** ensures pixel values stay in the valid [0, 1] range. During optimization, values might go slightly outside this range.

The images should look very similar because MSE directly tries to match pixels!

The goal is to demonstrate a key idea: **noisy pixels can be transformed into something meaningful simply by following a loss function** — in this case, just making the pixels look as close as possible to the target photo.

After 100 optimization steps with Adam, the loss drops close to zero and the generated image becomes practically identical to the content image. ✓

---

## Section 4: Viewing Progress

It's helpful to see how the image evolves during optimization. Let's create a callback that logs images at regular intervals.

In [ ]:
class ImageLogCB(Callback):
    """
    Callback that logs images during training and displays them at the end.
    
    Parameters:
    -----------
    log_every : int
        Save an image every this many iterations
    """
    # Run after ProgressCB (so progress bar updates first)
    order = ProgressCB.order + 1
    
    def __init__(self, log_every=10): 
        store_attr()         # Saves log_every as self.log_every
        self.images = []     # List to store logged images
        self.i = 0           # Counter for iterations
    
    def after_batch(self, learn):
        """
        Called after each batch/iteration.
        Saves the current image every log_every iterations.
        """
        # Check if we should log this iteration
        # i % log_every == 0 is true for i=0, log_every, 2*log_every, etc.
        if self.i % self.log_every == 0: 
            # Save a copy of the current predictions (image)
            # .clip(0, 1) keeps values in valid range
            # to_cpu moves to CPU to save GPU memory
            self.images.append(to_cpu(learn.preds.clip(0, 1)))
        self.i += 1  # Increment counter
    
    def after_fit(self, learn):
        """
        Called after training finishes.
        Displays all logged images in a grid.
        """
        show_images(self.images)

**What does the code above do?**

Creates a callback that:

1. **`__init__`**: Sets up logging frequency and empty image list

2. **`after_batch`**: After each optimization step:
   - Checks if we should log (every `log_every` steps)
   - If yes, saves a copy of the current image

3. **`after_fit`**: After all training:
   - Displays all saved images in a grid

**The modulo trick:**
```python
if self.i % self.log_every == 0:
```
This is true when `i` is divisible by `log_every`. Example with `log_every=30`:
- i=0: 0 % 30 = 0 (log)
- i=29: 29 % 30 = 29 (skip)
- i=30: 30 % 30 = 0 (log)
- i=60: 60 % 30 = 0 (log)

## What does `store_attr()` do?

`store_attr()` is a fastai utility that automatically does this:

```python
def __init__(self, log_every=10):
    self.log_every = log_every  # store_attr() does exactly this
    self.images = []
    self.i = 0
```

It inspects the `__init__` signature, finds all the parameters (`log_every` in this case), and assigns each one as `self.parameter_name = parameter_name`. It's just syntactic sugar to avoid writing repetitive `self.x = x` lines.

It becomes more useful when there are many parameters:

```python
# Without store_attr()
def __init__(self, log_every, start_epoch, save_path, verbose):
    self.log_every = log_every
    self.start_epoch = start_epoch
    self.save_path = save_path
    self.verbose = verbose

# With store_attr()
def __init__(self, log_every, start_epoch, save_path, verbose):
    store_attr()  # one line does all four assignments
```

In [ ]:
# Test the ImageLogCB with MSE loss
model = TensorModel(torch.rand_like(content_im))
learn = Learner(model, get_dummy_dls(150), loss_fn_mse, 
                lr=1e-2, cbs=cbs, opt_func=torch.optim.Adam)

# Fit with ImageLogCB that logs every 30 steps
# This will save images at steps 0, 30, 60, 90, 120, 150
learn.fit(1, cbs=[ImageLogCB(30)])

![](img_15.png)

**What does the code above do?**

Runs 150 optimization steps and shows the image at steps 0, 30, 60, 90, 120.

**What you should see:**
A sequence of images showing the random noise gradually becoming the face image. This is the power of gradient descent - it's "pulling" our random image toward the target!

## What happens when we run `learn.fit(1)`?

### One epoch = 150 steps here

Yes, there is only one epoch. But "epoch" here means something unusual because of `get_dummy_dls(150)`.

A normal epoch means *"go through the entire dataset once."* Here there is no real dataset — the dummy dataloader is just a counter that fires 150 times. So one epoch = 150 optimization steps. That's it.

---

### What `get_dummy_dls(150)` actually does

It creates a fake dataloader that yields 150 empty/dummy batches. There is **no real data** being loaded. Each "batch" is just a trigger that tells the `Learner` to run one optimization step.

The actual "data" is the image tensor living inside `TensorModel` itself — it gets updated in-place each step.

---

### What happens each of the 150 steps

```
Step 0:   predict() → get current image pixels
          get_loss() → MSE vs content_im
          backward() → gradients on pixels
          step()     → Adam nudges pixels slightly
          ImageLogCB saves image  ← (0 % 30 == 0)

Step 1–29: same loop, no image saved

Step 30:  same loop, ImageLogCB saves image  ← (30 % 30 == 0)

Step 60:  same loop, ImageLogCB saves image
Step 90:  same loop, ImageLogCB saves image
Step 120: same loop, ImageLogCB saves image
Step 149: same loop, no image saved
          (150 % 30 == 0 would be step 150, which doesn't exist)
```

So you get **5 snapshots** at steps 0, 30, 60, 90, 120 — showing the image progressively converging from noise toward `content_im`.

---

### No, there are NOT 150 copies of anything

The dummy dataloader doesn't copy or replicate data. It just ticks 150 times like a metronome. The only thing being modified across all 150 steps is the **single image tensor** inside `TensorModel`, which starts as random noise and gets refined step-by-step.

The optimizer runs 150 times — one `Adam.step()` per dummy batch. Each step slightly adjusts the pixel values of the image to reduce the MSE loss, and after all 150 steps the image looks very close to `content_im`.

---

## Section 5: Feature Extraction with VGG16

Now we get to the key insight of neural style transfer: using features from a pre-trained CNN.

**Why VGG16?**

VGG16 is a classic convolutional neural network that:
- Was trained on ImageNet (millions of images, 1000 categories)
- Has a simple, regular architecture (easy to extract features from)
- Has been shown to learn meaningful visual representations

### VGG16 Architecture

![VGG16 Architecture](https://neurohive.io/wp-content/uploads/2018/11/vgg16-1-e1542731207177.png)

VGG16 consists of:
- **Convolutional layers**: Extract features at different scales
- **Max pooling layers**: Reduce spatial size, increase receptive field
- **Fully connected layers**: For classification (we won't use these)

The "16" in VGG16 refers to the number of weight layers (13 conv + 3 FC).

In [ ]:
# List all VGG models available in timm
# timm is a library of pre-trained PyTorch image models
print(timm.list_models('*vgg*'))

**What does the code above do?**

Lists all models in `timm` that have "vgg" in their name:
- `vgg11`, `vgg13`, `vgg16`, `vgg19`: Standard VGG models
- `_bn` variants: With batch normalization
- `repvgg`: A modern variant

We'll use `vgg16` (without batch norm) as it's the most commonly used for style transfer.

In [ ]:
# Load pretrained VGG16 and get just the feature extraction part
# pretrained=True downloads weights trained on ImageNet
# .features gets just the convolutional layers (not the classifier)
# .to(def_device) moves to GPU if available
vgg16 = timm.create_model('vgg16', pretrained=True).to(def_device).features

**What does the code above do?**

1. **`timm.create_model('vgg16', pretrained=True)`**: Creates VGG16 with ImageNet-trained weights

2. **`.features`**: Gets only the convolutional part (we don't need the classifier)

3. **`.to(def_device)`**: Moves to GPU for faster computation

**Why just `.features`?**

VGG16 has two parts:
- `features`: Convolutional layers that extract visual features
- `classifier`: Fully connected layers for classification

We only need the feature extraction part!

### 🎮 Interactive: explore all 31 layers of `vgg16.features`

Instead of reading the printed architecture, click through it. The strip below is the real VGG16 `features` module (Conv → ReLU → MaxPool, five blocks). For any layer you get the output shape for our 256×256 input, the **receptive field** (how much of the photo one activation can see — the key to why deep layers capture *layout* rather than *pixels*), and a mock reconstruction showing what content loss at that layer can still pin down. The preset buttons load the exact `target_layers` this notebook uses: `(18, 25)`, the early `(1, 6)` experiment, and the 5-layer style set.

In [ ]:
# ============================================================================
# 🎮 INTERACTIVE -- run this cell. It loads ./interactive_viz/vgg_feature_explorer.html
# at full width, sized to fit the visualization (setup cell near the top required).
# Clickable ruler over all 31 layers of vgg16.features: output shapes,
# receptive fields, and what each depth responds to (edges -> parts -> layout).
# Preset buttons load the notebook's actual target_layers choices.
# ============================================================================
show_viz("interactive_viz/vgg_feature_explorer.html", height="1260px")

In [ ]:
# Uncomment to see the full VGG16 architecture
# vgg16

### Image Normalization for VGG16

VGG16 was trained on ImageNet images that were normalized with specific statistics. We need to apply the same normalization to our images.

**What is normalization?**

Normalization transforms data to have a specific mean and standard deviation:
```
normalized = (original - mean) / std
```

For ImageNet, the statistics were computed across millions of images:
- Mean: [0.485, 0.456, 0.406] for R, G, B channels
- Std: [0.229, 0.224, 0.225] for R, G, B channels

Using the same normalization ensures the VGG16 "sees" images the way it was trained to.

In [ ]:
# ImageNet normalization statistics
# These are the mean and standard deviation for each RGB channel
imagenet_mean = tensor([0.485, 0.456, 0.406])  # R, G, B means
imagenet_std = tensor([0.229, 0.224, 0.225])   # R, G, B stds

In [ ]:
# This won't work! Try uncommenting to see the error:
# (content_im - imagenet_mean) / imagenet_std

# The problem: shapes don't match for broadcasting

**Why doesn't the simple approach work?**

The shapes don't match for broadcasting:
- `content_im.shape`: (3, 256, 256)
- `imagenet_mean.shape`: (3,)

PyTorch doesn't know how to align these automatically.

In [ ]:
# The mean tensor has shape (3,) - just 3 values
imagenet_mean.shape

In [ ]:
# Our image has shape (3, 256, 256) - 3 channels, 256x256 pixels
content_im.shape

In [ ]:
# Solution: Reshape mean to (3, 1, 1) so it broadcasts correctly
# [:, None, None] adds two dimensions: (3,) -> (3, 1, 1)
imagenet_mean[:, None, None].shape

**Understanding Broadcasting:**

Broadcasting is how PyTorch handles operations between tensors of different shapes.

```
Image:        (3, 256, 256)   # 3 channels, 256x256 spatial
Mean:         (3,)            # Can't broadcast!
Mean reshaped:(3, 1, 1)       # Can broadcast!
```

With shape (3, 1, 1), the mean values are "stretched" across the spatial dimensions:
- mean[0] (red channel mean) is subtracted from all red pixels
- mean[1] (green channel mean) is subtracted from all green pixels
- mean[2] (blue channel mean) is subtracted from all blue pixels

### 🎮 Interactive: broadcasting, one stage at a time

The error above and its `.reshape(3, 1, 1)` fix are pure **broadcasting rules**: shapes align **right-to-left**, and only size-`1` dims stretch. Step through the four stages below — the clash, the reshape, the (free!) stretch, and the final `(im − mean) / std` — and watch the per-channel value ranges move from `[0, 1]` to the `[−2.12, +2.64]` the notebook prints.

In [ ]:
# ============================================================================
# 🎮 INTERACTIVE -- run this cell. It loads ./interactive_viz/vgg_normalization_broadcast.html
# at full width, sized to fit the visualization (setup cell near the top required).
# Why (im - imagenet_mean) crashes and .reshape(3,1,1) fixes it: 4 stages
# with the real RuntimeError, the right-to-left alignment rule, and the
# per-channel range bars moving from [0,1] to [-2.12, +2.64].
# ============================================================================
show_viz("interactive_viz/vgg_normalization_broadcast.html", height="820px")

In [ ]:
# Manual implementation of normalization
def normalize(im):
    """
    Normalize an image using ImageNet statistics.
    
    Parameters:
    -----------
    im : torch.Tensor
        Image tensor with shape (3, H, W) or (B, 3, H, W)
        
    Returns:
    --------
    torch.Tensor
        Normalized image with same shape
    """
    # These numerical values represent the mean and standard deviation of the ImageNet dataset 
    # for the RGB channels. Normalizing with these statistics shifts the input data to have 
    # zero mean and unit variance, aligning it with the distribution the pre-trained model 
    # was originally trained on for better feature extraction performance.

    # Reshape mean and std to (3, 1, 1) for proper broadcasting and move to the input's device
    imagenet_mean = tensor([0.485, 0.456, 0.406])[:, None, None].to(im.device)
    imagenet_std = tensor([0.229, 0.224, 0.225])[:, None, None].to(im.device)
    
    # Standard normalization formula: (x - mean) / std
    return (im - imagenet_mean) / imagenet_std

In [ ]:
# Check the range of normalized values
# Should be roughly centered around 0, typically in range [-3, 3]
normalize(content_im).min(), normalize(content_im).max()

(tensor(-2.1179, device='cuda:0'), tensor(2.6400, device='cuda:0'))

In [ ]:
# Check the mean of each channel after normalization
# dim=(1, 2) computes mean over spatial dimensions (height, width)
# Result shows average value for each of the 3 channels
normalize(content_im).mean(dim=(1, 2))

tensor([-0.9736, -0.9623, -0.4226], device='cuda:0')

In [ ]:
# Torchvision provides a built-in Normalize transform
# This is more convenient than our manual implementation
normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],  # ImageNet means
    std=[0.229, 0.224, 0.225]    # ImageNet stds
)

In [ ]:
# Verify it gives the same results
normalize(content_im).min(), normalize(content_im).max()

(tensor(-2.1179, device='cuda:0'), tensor(2.6400, device='cuda:0'))

---

## Section 6: Extracting Intermediate Features

The key to style transfer is extracting features from **intermediate layers** of VGG16, not just the final output.

**Why intermediate layers?**

Different layers capture different types of information:
- **Early layers (1-6)**: Edges, colors, simple textures
- **Middle layers (11-18)**: Patterns, textures, parts of objects
- **Late layers (25+)**: Object shapes, semantic content

In [ ]:
def calc_features(imgs, target_layers=(18, 25)):
    """
    Here we are looking at activations of layer 18 and 25
    Extract features from specific layers of VGG16.
    
    Parameters:
    -----------
    imgs : torch.Tensor
        Input image(s) with shape (3, H, W) or (B, 3, H, W)
    target_layers : tuple of int
        Which layer indices to extract features from
        
    Returns:
    --------
    list of torch.Tensor
        Feature maps from each target layer
    """
    # First, normalize the input image for VGG16
    x = normalize(imgs)
    
    # List to store features from target layers
    feats = []
    
    # Pass through VGG16 layers one by one
    # We only go up to max(target_layers)+1 (no need to compute further layers)
    for i, layer in enumerate(vgg16[:max(target_layers)+1]):
        # Apply this layer
        x = layer(x)
        
        # If this is one of our target layers, save the features
        if i in target_layers:
            feats.append(x.clone())  # .clone() creates a copy
    
    return feats

**When we say features, we just mean the activations of a layer.**

**What does the code above do?**

Passes an image through VGG16 and captures feature maps at specified layers:

1. **Normalize the image** for VGG16
2. **Iterate through layers** one at a time
3. **Save features** when we reach a target layer

**Example with target_layers=(18, 25):**
```
Layer 0 → Layer 1 → ... → Layer 18 (save!) → ... → Layer 25 (save!) → done
```

**Why `.clone()`?**

We clone the features because `x` keeps getting modified as we pass through layers. Without `.clone()`, we'd just save references that point to the same (final) tensor.

In [ ]:
# Test calc_features and examine the output shapes
feats = calc_features(content_im)
[f.shape for f in feats]

[torch.Size([512, 32, 32]), torch.Size([512, 16, 16])]

**What does the output mean?**

The shapes tell us about the feature maps:
- **Layer 18**: Shape (512, 32, 32) - 512 feature channels, 32x32 spatial resolution
- **Layer 25**: Shape (512, 16, 16) - 512 feature channels, 16x16 spatial resolution

Notice how:
- Spatial size decreases (256→32→16) as we go deeper (due to pooling)
- Number of channels increases (3→512) as we go deeper (more features)

Later layers have smaller spatial resolution but capture higher-level features!

In [ ]:
# Homework: Can you implement feature extraction using PyTorch hooks?
# Hooks are a more elegant way to capture intermediate activations

### Why Feature Extraction Matters

You may remember the article at https://distill.pub/2017/feature-visualization/ which shows how deep CNNs "learn" to see images.

**Key insight:**
- Early layers: Detect edges, colors, simple patterns
- Middle layers: Detect textures, repeating patterns
- Late layers: Detect objects, faces, semantic meaning

**For style transfer:**
- **Content**: Use late layer features (captures "what" is in the image)
- **Style**: Use early/middle layer features (captures "how" things look)

---

## Section 7: Content Loss with Perceptual Features

Now let's use VGG features instead of raw pixels to define our loss.

**Content Loss**: Measures how similar the high-level features of two images are.

Instead of comparing pixels directly (MSE), we compare the VGG features from later layers. This captures the "content" (structure, objects) without requiring exact pixel matches.

In [ ]:
class ContentLossToTarget():
    """
    Loss function that compares VGG features between an image and a target.
    
    This is also called "perceptual loss" - it measures perceptual similarity
    rather than pixel-by-pixel similarity.
    
    Parameters:
    -----------
    target_im : torch.Tensor
        The target content image
    target_layers : tuple of int
        Which VGG layers to extract features from
    """
    def __init__(self, target_im, target_layers=(18, 25)):
        # Store attributes
        fc.store_attr()
        
        # Pre-compute features of the target image
        # torch.no_grad() disables gradient tracking (we won't optimize the target)
        with torch.no_grad():
            self.target_features = calc_features(target_im, target_layers)
    
    def __call__(self, input_im):
        """
        Compute the content loss between input_im and the target.
        
        Parameters:
        -----------
        input_im : torch.Tensor
            The image being optimized
            
        Returns:
        --------
        loss : torch.Tensor
            Sum of MSE losses between features at each target layer
        """
        # Compute features of the input image
        input_features = calc_features(input_im, self.target_layers)
        
        # Sum MSE loss between features at each layer
        # zip pairs up corresponding features: (input_layer18, target_layer18), etc.
        # (f1-f2).pow(2).mean() is MSE between feature maps
        return sum((f1 - f2).pow(2).mean() 
                   for f1, f2 in zip(input_features, self.target_features))

**What does the code above do?**

Creates a loss function that:

1. **`__init__`**: Pre-computes the VGG features of the target image (done once)

2. **`__call__`**: For any input image:
   - Computes its VGG features
   - Calculates MSE between input and target features at each layer
   - Returns the sum of all these losses

**Why pre-compute target features?**

The target image never changes, so we only need to compute its features once. This saves computation during optimization.

**The loss formula:**
```
Content Loss = MSE(features_input_layer18, features_target_layer18) +
               MSE(features_input_layer25, features_target_layer25)
```

## Why the loop in `ContentLossToTarget.__call__`?

Because `target_layers=(18, 25)` — we're extracting features from **two** VGG layers, not one. So `calc_features` returns a list of two feature maps, and we need to compare both pairs.

The loop unpacks those pairs:

```python
# input_features  = [features_at_layer18, features_at_layer25]
# target_features = [features_at_layer18, features_at_layer25]

zip(input_features, self.target_features)
# → (input_layer18, target_layer18)   ← first pair
# → (input_layer25, target_layer25)   ← second pair
```

Then `sum(...)` adds both MSE losses into a single scalar that the optimizer can minimize.

---

## Why compare features from multiple layers?

Different VGG layers capture different levels of visual information:

- **Layer 18** — lower-level features: edges, textures, fine details
- **Layer 25** — higher-level features: shapes, object parts, broader structure

Comparing at only one layer would give an incomplete picture of perceptual similarity. By summing losses from both layers, you force the generated image to match the target at **multiple levels of abstraction** simultaneously.

---

## Contrast with the simple MSE loss earlier

```python
# Before: compare raw pixels — one single comparison
F.mse_loss(im, content_im)

# Now: compare VGG features at two layers — two comparisons, summed
sum((f1 - f2).pow(2).mean() for f1, f2 in zip(input_features, target_features))
```

The loop is what makes this **perceptual loss** rather than pixel loss — it's measuring *"do these images look similar to a neural network?"* rather than *"are these pixels identical?"*

---

## How `__call__` gets invoked

When you define `__call__` on a class, Python lets you use an **instance of that class as if it were a function**. So:

```python
# Create an instance
loss_fn = ContentLossToTarget(content_im, target_layers=(18, 25))

# Calling the instance like a function...
loss_fn(input_im)

# ...is exactly the same as:
loss_fn.__call__(input_im)
```

---

## Where it connects to the rest of the code

Remember from earlier, `ImageOptCB.get_loss` does:

```python
def get_loss(self, learn): learn.loss = learn.loss_func(learn.preds)
```

And `loss_fn` was passed into `Learner` as `loss_func`:

```python
loss_fn = ContentLossToTarget(content_im)

learn = Learner(model, get_dummy_dls(150), loss_fn, ...)
#                                          ^^^^^^^
#                                 stored as learn.loss_func
```

So when `get_loss` runs `learn.loss_func(learn.preds)`, it is literally doing:

```python
loss_fn(learn.preds)         # triggers __call__
# → ContentLossToTarget.__call__(learn.preds)
```

---

## The full chain

```
learn.fit(1)
    └─ get_loss()
        └─ learn.loss_func(learn.preds)
            └─ loss_fn(learn.preds)          # instance called like a function
                └─ __call__(input_im)         # Python routes here automatically
                    └─ compares VGG features, returns scalar loss
```

This is the same pattern as `loss_fn_mse` from before — except instead of a plain function, now the loss is a **class instance with state** (the pre-computed `target_features`). The `__call__` method is what makes the instance behave like a function so it plugs into the same `Learner` machinery seamlessly.

In [ ]:
# Create the perceptual/content loss function
loss_function_perceptual = ContentLossToTarget(content_im)

# Create a model starting from random noise
model = TensorModel(torch.rand_like(content_im))

# Set up and run training
learn = Learner(model, get_dummy_dls(150), loss_function_perceptual, 
                lr=1e-2, cbs=cbs, opt_func=torch.optim.Adam)
learn.fit(1, cbs=[ImageLogCB(log_every=30)])

![](img_16.png)

# Understanding `get_dummy_dls` and `TensorModel` in Neural Style Transfer

## The Problem

In neural style transfer, we're **not** training a neural network in the traditional sense. Instead, we're optimizing the **pixels of an image** directly. But fastai's `Learner` class expects a DataLoader with training data.

## The Solution

`get_dummy_dls(150)` creates a "fake" DataLoader that doesn't provide real data — it just tells the training loop how many iterations to run.

Here's what it likely looks like under the hood:

```python
def get_dummy_dls(n_batches):
    """
    Creates a dummy DataLoader that yields 'n_batches' empty batches.
    
    This tricks fastai's Learner into running n_batches iterations
    without needing actual training data.
    """
    # Create a simple dataset of the right length
    dummy_data = [(None, None) for _ in range(n_batches)]
    
    # Wrap in a DataLoader
    dl = DataLoader(dummy_data, batch_size=1)
    
    # fastai expects a DataLoaders object (train + valid)
    return DataLoaders(dl, dl)
```

## How Does It Use the Image Instead of the DataLoader?

The magic is in `TensorModel`. It **ignores** the DataLoader input entirely and just returns the image pixels as learnable parameters.

### How `TensorModel` Works

```python
class TensorModel(nn.Module):
    def __init__(self, image):
        super().__init__()
        # The image pixels become learnable parameters!
        self.image = nn.Parameter(image.clone())
    
    def forward(self, *args):  # <-- ignores any input!
        return self.image      # <-- just returns the image
```

### The Training Loop (Simplified)

```python
# What fastai's Learner does each iteration:

for batch in dummy_dataloader:      # ① Get "fake" batch (ignored!)
    
    output = model()                 # ② Model returns image pixels
                                     #    (input from dataloader is ignored)
    
    loss = combined_loss(output)     # ③ Compute style + content loss
    
    loss.backward()                  # ④ Compute gradients w.r.t. pixels
    
    optimizer.step()                 # ⑤ Update pixel values!
```

## Comparison: Traditional Training vs Style Transfer

| Aspect | Traditional Training | Style Transfer |
|--------|---------------------|----------------|
| **DataLoader provides** | Real images/labels | Nothing useful (dummy) |
| **Model input** | Batch from DataLoader | Ignored |
| **Model output** | Predictions | The image pixels themselves |
| **What gets updated** | Network weights | Pixel values |

## Summary

The model **ignores** whatever the DataLoader provides. It simply returns its internal `nn.Parameter` (the image), which is what gets optimized. The dummy DataLoader only controls **how many times** the optimization loop runs.

The `150` in `get_dummy_dls(150)` means: run 150 optimization steps to iteratively refine the image.

**What should you see?**

The image evolves differently than with MSE loss:
- The **structure** and **shapes** emerge (face shape, features)
- But the **exact details** and **colors** may differ from the original

This is because we're matching high-level features, not pixels!

### The Impact of Layer Choice

Choosing different layers changes what kind of features are preserved. Let's try earlier layers.

In [ ]:
# Use early layers (1, 6) instead of late layers (18, 25)
# Early layers capture textures and colors, not high-level structure
loss_function_perceptual = ContentLossToTarget(content_im, target_layers=(1, 6))

model = TensorModel(torch.rand_like(content_im))
learn = Learner(model, get_dummy_dls(150), loss_function_perceptual, 
                lr=1e-2, cbs=cbs, opt_func=torch.optim.Adam)
learn.fit(1, cbs=[ImageLogCB(log_every=30)])

![](img_17.png)

# Understanding `get_dummy_dls` and `TensorModel` in Neural Style Transfer

## The Problem

In neural style transfer, we're **not** training a neural network in the traditional sense. Instead, we're optimizing the **pixels of an image** directly. But fastai's `Learner` class expects a DataLoader with training data.

## The Solution

`get_dummy_dls(150)` creates a "fake" DataLoader that doesn't provide real data — it just tells the training loop how many iterations to run.

Here's what it likely looks like under the hood:

```python
def get_dummy_dls(n_batches):
    """
    Creates a dummy DataLoader that yields 'n_batches' empty batches.
    
    This tricks fastai's Learner into running n_batches iterations
    without needing actual training data.
    """
    # Create a simple dataset of the right length
    dummy_data = [(None, None) for _ in range(n_batches)]
    
    # Wrap in a DataLoader
    dl = DataLoader(dummy_data, batch_size=1)
    
    # fastai expects a DataLoaders object (train + valid)
    return DataLoaders(dl, dl)
```

## How Does It Use the Image Instead of the DataLoader?

The magic is in `TensorModel`. It **ignores** the DataLoader input entirely and just returns the image pixels as learnable parameters.

### How `TensorModel` Works

```python
class TensorModel(nn.Module):
    def __init__(self, image):
        super().__init__()
        # The image pixels become learnable parameters!
        self.image = nn.Parameter(image.clone())
    
    def forward(self, *args):  # <-- ignores any input!
        return self.image      # <-- just returns the image
```

### The Training Loop (Simplified)

```python
# What fastai's Learner does each iteration:

for batch in dummy_dataloader:      # ① Get "fake" batch (ignored!)
    
    output = model()                 # ② Model returns image pixels
                                     #    (input from dataloader is ignored)
    
    loss = combined_loss(output)     # ③ Compute style + content loss
    
    loss.backward()                  # ④ Compute gradients w.r.t. pixels
    
    optimizer.step()                 # ⑤ Update pixel values!
```

## Comparison: Traditional Training vs Style Transfer

| Aspect | Traditional Training | Style Transfer |
|--------|---------------------|----------------|
| **DataLoader provides** | Real images/labels | Nothing useful (dummy) |
| **Model input** | Batch from DataLoader | Ignored |
| **Model output** | Predictions | The image pixels themselves |
| **What gets updated** | Network weights | Pixel values |

## Summary

The model **ignores** whatever the DataLoader provides. It simply returns its internal `nn.Parameter` (the image), which is what gets optimized. The dummy DataLoader only controls **how many times** the optimization loop runs.

The `150` in `get_dummy_dls(150)` means: run 150 optimization steps to iteratively refine the image.

**What's different?**

With early layers (1, 6):
- **Colors** and **textures** are matched more closely
- **High-level structure** might be less clear

With late layers (18, 25):
- **Structure** and **shapes** are preserved
- **Colors** and **fine details** may vary

**This is the key insight for style transfer:**
- Use **late layers** for content (structure)
- Use **early layers** for style (texture)

---

## Section 8: Style Loss with Gram Matrix

Now for the clever part: how do we capture **style** (texture, patterns) independent of spatial location?

**The problem:**
If we just compare feature maps directly, we're comparing "where" features appear. But style is about "what kinds" of features appear together, not where.

![](img_18.png)

### Why Feature Maps Alone Don't Work for Style

Feature maps encode information **spatially**:
- A feature at position (10, 20) captures patterns at that location
- If we shift the image, the feature moves too

For style, we want to know:
- What **kinds** of features are present?
- Which features tend to **occur together**?
- But **not** where they are located!

```
Feature Map (spatial):
┌─────────────────┐
│ 0  1  0  2  0   │  ← Feature 1 values at each position
│ 1  3  2  0  1   │
│ 0  2  4  1  0   │
└─────────────────┘

For style, we don't care WHERE values are,
just what patterns of values occur.
```

### The Gram Matrix Solution

The **Gram Matrix** captures which features occur together by computing **correlations** between all pairs of features.

**How it works:**
1. Take a feature map with F features and H×W spatial dimensions
2. Flatten the spatial dimensions: (F, H×W)
3. Compute the dot product of this matrix with its transpose
4. Result: F×F matrix where entry (i,j) measures correlation between features i and j

**The key property:**
The Gram matrix is **translation invariant** - moving the image doesn't change which features co-occur!

**Visual explanation of Gram Matrix:**

```
Feature Map (F features, H×W spatial):
         Spatial Position (flattened to HW)
            1  2  3  4  5  6  7  8  9
Feature 1: [0, 1, 0, 1, 1, 0, 0, 1, 1]  (appears at positions 2,4,5,8,9)
Feature 2: [0, 1, 0, 1, 0, 0, 0, 0, 1]  (appears at positions 2,4,9)
Feature 3: [1, 0, 1, 1, 1, 1, 1, 1, 0]  (appears at positions 1,3,4,5,6,7,8)
Feature 4: [1, 0, 1, 1, 0, 1, 1, 0, 0]  (appears at positions 1,3,4,6,7)

Gram Matrix = Feature × Feature^T

         F1  F2  F3  F4
    F1 [ 5   3   3   1 ]  ← F1·F1=5 (appears 5 times)
    F2 [ 3   3   1   1 ]     F1·F2=3 (co-occur 3 times)
    F3 [ 3   1   7   5 ]
    F4 [ 1   1   5   5 ]
```

Entry (i,j) tells us how often features i and j appear together!

![](img_19.png)

In [ ]:
# Recreating the diagram example in code
# 4 features, 9 spatial positions (3x3 flattened)
t = tensor([[0, 1, 0, 1, 1, 0, 0, 1, 1],  # Feature 1
            [0, 1, 0, 1, 0, 0, 0, 0, 1],  # Feature 2
            [1, 0, 1, 1, 1, 1, 1, 1, 0],  # Feature 3
            [1, 0, 1, 1, 0, 1, 1, 0, 0]]) # Feature 4

In [ ]:
# Compute Gram matrix using einsum
# 'fs, gs -> fg' means:
# - f: first dimension of first tensor (features)
# - s: second dimension (spatial, summed over)
# - g: first dimension of second tensor (features)
# Result: for each pair (f, g), sum over s of t[f,s] * t[g,s]
torch.einsum('fs, gs -> fg', t, t)

**Understanding `torch.einsum`:**

`einsum` (Einstein summation) is a powerful way to express tensor operations.

The string `'fs, gs -> fg'` means:
- First tensor has indices `f` and `s`
- Second tensor has indices `g` and `s`
- Output has indices `f` and `g`
- Index `s` appears in both inputs but not output, so it's summed over

This computes:
```
output[f, g] = sum over s of (input1[f, s] * input2[g, s])
```

Which is exactly the Gram matrix!

## Why `fs` and `gs` and not `fs` and `sf`?

The confusion is completely understandable. Let me untangle it.

---

## What einsum notation actually means

In `torch.einsum('fs, gs -> fg', t, t)`:

- `f` = row index of the **first** `t`
- `s` = column index of the **first** `t`
- `g` = row index of the **second** `t`
- `s` = column index of the **second** `t` ← **same letter = sum over this dimension**

The rule in einsum is: **any letter that appears on the left but not on the right gets summed over.** Here `s` disappears in the output `fg`, so einsum automatically sums over all spatial positions.

---

## This IS matrix × transpose

Let's compare manually:

```
t       has shape [4 features, 9 spatial]   → indices: f, s
t.T     has shape [9 spatial, 4 features]   → indices: s, g
t @ t.T has shape [4 features, 4 features]  → indices: f, g
```

In standard matrix multiplication `t @ t.T`, what happens at position `[f, g]`?

```
result[f, g] = sum over s of:  t[f, s] * t.T[s, g]
                                           ↑
                              t.T[s,g] is just t[g,s]

so: result[f, g] = sum over s of:  t[f, s] * t[g, s]
```

Now look at einsum `'fs, gs -> fg'`:

```
result[f, g] = sum over s of:  t[f, s] * t[g, s]
```

**Identical.** The einsum is doing exactly the same computation.

---

## Why `gs` and not `sf` for the transpose?

This is the key insight. In `t @ t.T` you physically flip the matrix. In einsum **you don't need to flip anything** — you just assign different row labels to the two copies of `t`.

```python
# Standard: you must physically transpose t
t @ t.T          # t.T is a new tensor with swapped axes

# Einsum: no transposing needed
# just say "this copy's rows are called g, not f"
torch.einsum('fs, gs -> fg', t, t)
```

By writing `gs` instead of `sf`, you're telling einsum: *"treat the rows of this second copy independently from the first copy's rows"* — which is exactly what transposing achieves, but without the extra operation.

---

## One-line summary

`fs, gs` uses **different first letters** (`f` vs `g`) to say *"these are two independent row indices"* and **same second letter** (`s`) to say *"sum over this shared dimension"* — which is precisely what matrix × transpose does, just expressed more explicitly.

In [ ]:
# Alternative: matrix multiplication with transpose
# t @ t.T is the same as einsum('fs, gs -> fg', t, t)
t.matmul(t.T)

**Two ways to compute Gram matrix:**

1. **`torch.einsum('fs, gs -> fg', t, t)`**: More explicit about dimensions
2. **`t.matmul(t.T)`** or **`t @ t.T`**: Standard matrix multiplication

Both give the same result! Use whichever is clearer to you.

### 🎮 Interactive: build a Gram matrix by hand

This is the single most important idea in the notebook, so here it is at toy scale: **4 channels of 3×3 feature maps → a 4×4 Gram matrix.** Watch each map flatten into a row, then every row get dotted with every row. **Click any cell of G to compute it**, toggle between the `einsum` and `f @ f.T` code (identical math), switch the `÷ (H·W)` normalization on and off — and then press **🔀 Shuffle pixel positions**: every feature map scrambles, but G doesn't change by a single digit. That invariance *is* why Gram matrices capture style but not layout.

In [ ]:
# ============================================================================
# 🎮 INTERACTIVE -- run this cell. It loads ./interactive_viz/gram_matrix_lab.html
# at full width, sized to fit the visualization (setup cell near the top required).
# Toy Gram matrix: 4 channels x (3x3) -> 4x4 G. Stages: flatten -> pick pair
# -> multiply & sum -> write G[c,d]. Click any G cell to compute it; toggle
# einsum vs f@f.T; press 'Shuffle pixels' to PROVE spatial invariance.
# ============================================================================
show_viz("interactive_viz/gram_matrix_lab.html", height="790px")

### Computing Gram Matrices for Style Transfer

Now let's load a style image and compute its Gram matrices.

In [ ]:
# Download and display the style image (spiderweb)
style_im = download_image(spiderweb_url).to(def_device)
show_image(style_im);

![](img_20.png)

In [ ]:
def calc_grams(img, target_layers=(1, 6, 11, 18, 25)):
    """
    Compute Gram matrices for features at specified VGG layers.
    
    Parameters:
    -----------
    img : torch.Tensor
        Input image
    target_layers : tuple of int
        VGG layer indices to extract features from
        
    Returns:
    --------
    L (list)
        List of Gram matrices, one per layer
    """
    # L is a fastcore enhanced list
    return L(
        # For each feature map x from calc_features:
        # Compute Gram matrix using einsum
        # 'chw, dhw -> cd' means:
        #   c, d: channel dimensions (features)
        #   h, w: spatial dimensions (summed over)
        # Divide by (h*w) to normalize by spatial size
        torch.einsum('chw, dhw -> cd', x, x) / (x.shape[-2] * x.shape[-1])
        for x in calc_features(img, target_layers)
    )

**What does the code above do?**

For each layer's feature map:
1. **Extract features** with `calc_features`
2. **Compute Gram matrix** with einsum
3. **Normalize** by dividing by spatial size (h×w)

**Why normalize by spatial size?**

Without normalization, layers with larger spatial dimensions would dominate the loss. Dividing by (h×w) makes the loss comparable across layers.

**The einsum string `'chw, dhw -> cd'`:**
- `c`, `d`: Feature/channel dimensions (kept in output)
- `h`, `w`: Height and width (summed over)
- Result: Gram matrix of shape (C, C)

### Why divide by `x.shape[-2] * x.shape[-1]`?

The raw Gram matrix values grow proportionally with the number of spatial locations (height × width) — a larger image produces larger raw values simply because there are more positions being summed over, not because the style is actually different.

Dividing by `h * w` converts this from an **absolute** measure into a **relative** one:

```python
torch.einsum('chw, dhw -> cd', x, x) / (x.shape[-2] * x.shape[-1])
#                                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^
#                                        normalize by spatial size
```

This ensures that style loss comparisons remain **consistent across different image resolutions** — the loss won't explode or change meaning simply because one image is larger or smaller than another.

### Channels as Features

Each channel in a VGG feature map represents a different learned filter — one might detect diagonal edges, another detects blobs of color, another detects fur-like textures, and so on. So for a given layer, if there are 256 channels, you have 256 features.

The Gram matrix then captures **how correlated these 256 features are with each other** across all spatial positions — which is what encodes "style." It answers questions like: *"whenever this edge-detector fires, does this texture-detector also tend to fire?"*

That co-occurrence pattern is style — and it's completely independent of **where** in the image those features activate, which is why the spatial dimensions `h, w` get summed over and disappear in the output `cd`.

# Understanding `calc_grams`: Computing Gram Matrices for Style Transfer

## The Function

```python
def calc_grams(img, target_layers=(1, 6, 11, 18, 25)):
    """
    Compute Gram matrices for features at specified VGG layers.
    
    Parameters:
    -----------
    img : torch.Tensor
        Input image
    target_layers : tuple of int
        VGG layer indices to extract features from
        
    Returns:
    --------
    L (list)
        List of Gram matrices, one per layer
    """
    # L is a fastcore enhanced list
    return L(
        # For each feature map x from calc_features:
        # Compute Gram matrix using einsum
        # 'chw, dhw -> cd' means:
        #   c, d: channel dimensions (features)
        #   h, w: spatial dimensions (summed over)
        # Divide by (h*w) to normalize by spatial size
        torch.einsum('chw, dhw -> cd', x, x) / (x.shape[-2] * x.shape[-1])
        for x in calc_features(img, target_layers)
    )
```

---

## Yes! Each Element is a 2D Tensor

Each element of the returned list will be a **2D tensor** (the Gram matrix).

---

## What `calc_features` Returns

For `target_layers=(1, 6, 11, 18, 25)`, `calc_features` returns a list of 5 feature maps:

```python
features = calc_features(img, target_layers=(1, 6, 11, 18, 25))

len(features)  # 5 (one for each target layer)
```

Each feature map has shape `(Channels, Height, Width)`:

| Index | Layer | Shape | Description |
|-------|-------|-------|-------------|
| `features[0]` | Layer 1 | `(64, 256, 256)` | 64 feature maps, full resolution |
| `features[1]` | Layer 6 | `(128, 128, 128)` | 128 feature maps, after 1 pooling |
| `features[2]` | Layer 11 | `(256, 64, 64)` | 256 feature maps, after 2 poolings |
| `features[3]` | Layer 18 | `(512, 32, 32)` | 512 feature maps, after 3 poolings |
| `features[4]` | Layer 25 | `(512, 16, 16)` | 512 feature maps, after 4 poolings |

---

## What `calc_grams` Returns

Each 3D feature map `(C, H, W)` becomes a 2D Gram matrix `(C, C)`:

```python
grams = calc_grams(img, target_layers=(1, 6, 11, 18, 25))

len(grams)  # 5 (one Gram matrix per layer)
```

| Index | From Layer | Input Shape | Gram Matrix Shape |
|-------|------------|-------------|-------------------|
| `grams[0]` | Layer 1 | `(64, 256, 256)` | `(64, 64)` |
| `grams[1]` | Layer 6 | `(128, 128, 128)` | `(128, 128)` |
| `grams[2]` | Layer 11 | `(256, 64, 64)` | `(256, 256)` |
| `grams[3]` | Layer 18 | `(512, 32, 32)` | `(512, 512)` |
| `grams[4]` | Layer 25 | `(512, 16, 16)` | `(512, 512)` |

---

## How the Gram Matrix is Computed

```python
torch.einsum('chw, dhw -> cd', x, x) / (x.shape[-2] * x.shape[-1])
```

### Step-by-Step for Layer 1 Features

```
Input: x with shape (64, 256, 256)
       └─┘  └──────┘
        C    H × W

einsum 'chw, dhw -> cd':
- c and d are channel indices (0 to 63)
- h and w are spatial indices (summed over)
- For each pair (c, d): sum over all h,w positions of x[c,h,w] * x[d,h,w]

Output: (64, 64) matrix
        └─────┘
         C × C

Normalize by: H × W = 256 × 256 = 65,536
```

### Visual Representation

```
Feature Map x: shape (C, H, W)
┌─────────────────────────────┐
│  Channel 0: [H × W pixels]  │──┐
│  Channel 1: [H × W pixels]  │──┼──┐
│  Channel 2: [H × W pixels]  │──┼──┼──┐
│  ...                        │  │  │  │
│  Channel C-1: [H × W pixels]│  │  │  │
└─────────────────────────────┘  │  │  │
                                 │  │  │
         Gram Matrix: (C × C)    │  │  │
        ┌────────────────────────┼──┼──┼─────────┐
        │                        ▼  ▼  ▼         │
        │        C0   C1   C2  ...  C(C-1)       │
        │   C0 [ corr corr corr ... corr ]       │
        │   C1 [ corr corr corr ... corr ]       │
        │   C2 [ corr corr corr ... corr ]       │
        │   ... [ ...  ...  ... ... ...  ]       │
        │ C(C-1)[ corr corr corr ... corr ]      │
        └────────────────────────────────────────┘
        
        Each entry = correlation between two channels
                     (summed over all spatial positions)
```

### What Each Gram Entry Means

```python
# Gram[i, j] = how much do channel i and channel j activate together?

# If channel 3 detects "horizontal edges" and channel 7 detects "blue color"
# Then Gram[3, 7] measures: "How often do horizontal edges appear with blue?"

# This captures STYLE without caring WHERE these patterns appear!
```

---

## Summary

```
calc_features(img, (1, 6, 11, 18, 25))
        │
        ▼
┌─────────────────────────────────────────────────────────┐
│ features[0]: (64, 256, 256)   ──► Gram: (64, 64)        │
│ features[1]: (128, 128, 128)  ──► Gram: (128, 128)      │
│ features[2]: (256, 64, 64)    ──► Gram: (256, 256)      │
│ features[3]: (512, 32, 32)    ──► Gram: (512, 512)      │
│ features[4]: (512, 16, 16)    ──► Gram: (512, 512)      │
└─────────────────────────────────────────────────────────┘
        │
        ▼
calc_grams returns: [5 Gram matrices, all 2D tensors]
```

The spatial dimensions `(H, W)` are **eliminated** by the einsum—only channel correlations remain. This is why Gram matrices capture **style** (texture patterns) independent of **location**.

In [ ]:
# Compute Gram matrices for the style image
style_grams = calc_grams(style_im)

In [ ]:
# Check the shapes of Gram matrices at each layer
[g.shape for g in style_grams]

![](img_21.png)

**Understanding the Gram matrix shapes:**

| Layer | Feature Shape | Gram Shape |
|-------|---------------|------------|
| 1 | (64, H, W) | (64, 64) |
| 6 | (128, H, W) | (128, 128) |
| 11 | (256, H, W) | (256, 256) |
| 18 | (512, H, W) | (512, 512) |
| 25 | (512, H, W) | (512, 512) |

The Gram matrix size depends only on the number of features (channels), not the spatial size. This is how we remove spatial information!

### 🎮 Interactive (3D): the tensor the Gram matrix eats

The shapes above (`(64, 64)`, …, `(512, 512)`) come from collapsing a 3-D activation tensor. Below is that tensor in actual 3-D: a stack of C channel planes (**drag to rotate**). **Click one plane, then another** — their element-wise product flashes and collapses into a single purple cell of G. Diagonal cells are a channel's own energy; off-diagonals measure *co-firing*. Note how H×W vanishes completely — G's size depends only on C, which is exactly why every layer's Gram is square.

In [ ]:
# ============================================================================
# 🎮 INTERACTIVE -- run this cell. It loads ./interactive_viz/gram_3d_channels.html
# at full width, sized to fit the visualization (setup cell near the top required).
# three.js: drag-rotate a stack of 6 channel planes (8x8 each); click two
# planes to watch their product collapse into one cell of the 6x6 Gram.
# NOTE: needs internet access (loads three.js from cdnjs).
# ============================================================================
show_viz("interactive_viz/gram_3d_channels.html", height="1030px")

In [ ]:
# Fastcore's L has convenient methods like .attrgot
# This gets the 'shape' attribute of each element
style_grams.attrgot('shape')

In [ ]:
class StyleLossToTarget():
    """
    Loss function that compares Gram matrices between an image and a style target.
    
    This measures style similarity by comparing which features co-occur,
    ignoring where they occur spatially.
    
    Parameters:
    -----------
    target_im : torch.Tensor
        The style image
    target_layers : tuple of int
        VGG layers to use for style (default uses multiple layers)
    """
    def __init__(self, target_im, target_layers=(1, 6, 11, 18, 25)):
        fc.store_attr()
        # Pre-compute Gram matrices of the style image
        with torch.no_grad(): 
            self.target_grams = calc_grams(target_im, target_layers)
    
    def __call__(self, input_im):
        """
        Compute style loss between input and target.
        
        Returns:
        --------
        loss : torch.Tensor
            Sum of MSE between Gram matrices at each layer
        """
        # Compute Gram matrices of input image
        input_grams = calc_grams(input_im, self.target_layers)
        
        # Sum MSE loss between Gram matrices at each layer
        return sum((f1 - f2).pow(2).mean() 
                   for f1, f2 in zip(input_grams, self.target_grams))

**What does the code above do?**

Creates a style loss function that:

1. **`__init__`**: Pre-computes Gram matrices of the style image

2. **`__call__`**: For any input image:
   - Computes its Gram matrices
   - Calculates MSE between input and target Grams at each layer
   - Returns the sum

**Key difference from content loss:**
- Content loss: Compares **features directly** (spatial info preserved)
- Style loss: Compares **Gram matrices** (spatial info removed)

In [ ]:
# Create style loss function
style_loss = StyleLossToTarget(style_im)

In [ ]:
# Test: compute style loss between content and style images
# Higher value = more different styles
style_loss(content_im)

tensor(501.6082, device='cuda:0', grad_fn=<AddBackward0>)

## `ContentLossToTarget` vs `StyleLossToTarget`

---

## What each one compares

| | `ContentLossToTarget` | `StyleLossToTarget` |
|---|---|---|
| Compares | Raw feature maps | Gram matrices of feature maps |
| Captures | **What** is in the image | **How** the image looks/feels |
| Layers used | `(18, 25)` — deeper layers | `(1, 6, 11, 18, 25)` — all layers |
| Question asked | "Do these images show the same objects?" | "Do these images have the same texture/style?" |

---

## Content loss — preserving structure

```python
self.target_features = calc_features(target_im, target_layers)
# stores raw feature maps: shape [channels, height, width]
```

Raw feature maps retain **spatial information** — which features activate **where**. Comparing them directly asks: *"are the same things happening at the same locations?"*

This is what preserves the **content** — the lady's face stays in the same position, the sunglasses stay where they are.

---

## Style loss — preserving texture

```python
self.target_grams = calc_grams(target_im, target_layers)
# stores Gram matrices: shape [channels, channels]
```

Gram matrices **throw away spatial information** entirely by summing over `h, w`. What remains is only the correlation between channels — *"do these features tend to co-occur?"* — with no memory of where.

This is what captures **style** — the brushstroke patterns, color relationships, and textures of a Van Gogh painting, without caring where exactly each stroke is.

---

## Why use all 5 layers for style but only 2 for content?

Style needs to be captured at **every scale**:
- Early layers `(1, 6)` → fine textures, brushstroke thickness
- Middle layers `(11, 18)` → color patterns, broader strokes
- Late layer `(25)` → overall tonal relationships

Content only needs **deep layers** `(18, 25)` because those capture high-level semantics (faces, objects) rather than low-level pixel details. Using early layers for content would force pixel-level similarity, making the result look like a bad photocopy rather than a stylized painting.

---

## Why do we need both?

Neither loss alone produces good style transfer:

- **Content loss only** → the generated image converges back to looking exactly like the content photo (just like the MSE experiment earlier)
- **Style loss only** → the generated image gets the right textures but loses all structure, becoming an abstract blob of style with no recognizable content

Together, they pull in opposite directions and find a balance: **the content of one image rendered in the style of another.**

---

## Section 9: Style Transfer - Combining Content and Style

Finally! Let's combine content loss and style loss to perform actual style transfer.

In [ ]:
# Start from the content image (instead of random noise)
# This helps preserve content structure
model = TensorModel(content_im)

# Create loss functions
style_loss = StyleLossToTarget(style_im)      # Compare style to spiderweb
content_loss = ContentLossToTarget(content_im) # Compare content to face

def combined_loss(x):
    """
    Combined loss = style loss + content loss
    
    Minimizing this:
    - Makes the image have similar style to style_im (texture, patterns)
    - Makes the image have similar content to content_im (structure, shapes)
    """
    return style_loss(x) + content_loss(x)

# Train
learn = Learner(model, get_dummy_dls(150), combined_loss, 
                lr=1e-2, cbs=cbs, opt_func=torch.optim.Adam)
learn.fit(1, cbs=[ImageLogCB(30)])

![](img_22.png)

# Understanding `get_dummy_dls` and `TensorModel` in Neural Style Transfer

## The Problem

In neural style transfer, we're **not** training a neural network in the traditional sense. Instead, we're optimizing the **pixels of an image** directly. But fastai's `Learner` class expects a DataLoader with training data.

## The Solution

`get_dummy_dls(150)` creates a "fake" DataLoader that doesn't provide real data — it just tells the training loop how many iterations to run.

Here's what it likely looks like under the hood:

```python
def get_dummy_dls(n_batches):
    """
    Creates a dummy DataLoader that yields 'n_batches' empty batches.
    
    This tricks fastai's Learner into running n_batches iterations
    without needing actual training data.
    """
    # Create a simple dataset of the right length
    dummy_data = [(None, None) for _ in range(n_batches)]
    
    # Wrap in a DataLoader
    dl = DataLoader(dummy_data, batch_size=1)
    
    # fastai expects a DataLoaders object (train + valid)
    return DataLoaders(dl, dl)
```

## How Does It Use the Image Instead of the DataLoader?

The magic is in `TensorModel`. It **ignores** the DataLoader input entirely and just returns the image pixels as learnable parameters.

### How `TensorModel` Works

```python
class TensorModel(nn.Module):
    def __init__(self, image):
        super().__init__()
        # The image pixels become learnable parameters!
        self.image = nn.Parameter(image.clone())
    
    def forward(self, *args):  # <-- ignores any input!
        return self.image      # <-- just returns the image
```

### The Training Loop (Simplified)

```python
# What fastai's Learner does each iteration:

for batch in dummy_dataloader:      # ① Get "fake" batch (ignored!)
    
    output = model()                 # ② Model returns image pixels
                                     #    (input from dataloader is ignored)
    
    loss = combined_loss(output)     # ③ Compute style + content loss
    
    loss.backward()                  # ④ Compute gradients w.r.t. pixels
    
    optimizer.step()                 # ⑤ Update pixel values!
```

## Comparison: Traditional Training vs Style Transfer

| Aspect | Traditional Training | Style Transfer |
|--------|---------------------|----------------|
| **DataLoader provides** | Real images/labels | Nothing useful (dummy) |
| **Model input** | Batch from DataLoader | Ignored |
| **Model output** | Predictions | The image pixels themselves |
| **What gets updated** | Network weights | Pixel values |

## Summary

The model **ignores** whatever the DataLoader provides. It simply returns its internal `nn.Parameter` (the image), which is what gets optimized. The dummy DataLoader only controls **how many times** the optimization loop runs.

The `150` in `get_dummy_dls(150)` means: run 150 optimization steps to iteratively refine the image.

**What does the code above do?**

1. **Start from content image**: Instead of random noise, we begin with the face photo. This helps preserve the content structure.

2. **Create both loss functions**:
   - `style_loss`: Measures style difference from spiderweb
   - `content_loss`: Measures content difference from face

3. **Combined loss**: Sum of both losses

4. **Optimization**: Finds an image that balances both constraints

**The result:**
An image that has the **structure of the face** but the **texture/style of the spiderweb**!

In [ ]:
# View the final stylized result
show_image(learn.model().clip(0, 1));

![](img_23.png)

### Experimenting with Parameters

Let's try:
- Starting from random noise (instead of content image)
- Weighting style loss lower (so content dominates more)
- Using different layers for content

In [ ]:
# Start from random noise
model = TensorModel(torch.rand_like(content_im))

# New loss functions with different settings
style_loss = StyleLossToTarget(style_im)
content_loss = ContentLossToTarget(content_im, target_layers=(6, 18, 25))  # More layers

def combined_loss(x):
    # Weight style loss by 0.2 (reduce its importance)
    # This means content will be preserved more strongly
    return style_loss(x) * 0.2 + content_loss(x)

# More iterations, higher learning rate
learn = Learner(model, get_dummy_dls(300), combined_loss, 
                lr=5e-2, cbs=cbs, opt_func=torch.optim.Adam)
learn.fit(1, cbs=[ImageLogCB(60)])

**What's different in this experiment?**

1. **Random starting point**: Tests if we can reconstruct from nothing

2. **`style_loss(x) * 0.2`**: Reduces style weight
   - Higher style weight → more texture, less recognizable content
   - Lower style weight → clearer content, less stylization

3. **More content layers `(6, 18, 25)`**: Captures content at multiple scales

4. **More iterations (300)**: More time to optimize

5. **Higher learning rate (5e-2)**: Faster changes per step

**Try experimenting with these parameters!**
- What happens with `style_loss(x) * 2`?
- What if you only use layer 25 for content?
- What about different style images?

### 🎮 Interactive: the playground for these experiments

The last few cells all pulled the same three levers: **starting point** (content copy vs random noise), **style-loss weight** (e.g. `style_loss(x) * 0.2` — try the slider), and **which layers guard the content** (`(18, 25)` vs `(1, 6)`). The simulation below lets you re-run the whole sweep instantly: press ▶, watch the content and style losses fall at different speeds, and **click the loss plot or the iteration chips** to jump anywhere in the run. (It's a hand-built intuition machine, not a real VGG — the code panel shows what the real run would be.)

In [ ]:
# ============================================================================
# 🎮 INTERACTIVE -- run this cell. It loads ./interactive_viz/style_transfer_playground.html
# at full width, sized to fit the visualization (setup cell near the top required).
# Simulation of the notebook's final experiments: start from content vs noise,
# style-weight slider, early-vs-late content layers, dual loss curves.
# CLICK the loss plot or iteration chips to jump to any point in the run.
# ============================================================================
show_viz("interactive_viz/style_transfer_playground.html", height="820px")

---

## Section 10: Comparison - Without miniai Framework

For educational purposes, here's what the same style transfer looks like without our miniai training framework. This shows the "raw" PyTorch approach.

In [ ]:
# ============================================================================
# STYLE TRANSFER WITHOUT MINIAI (Raw PyTorch)
# ============================================================================

# Step 1: Create the image to optimize (random noise)
im = torch.rand(3, 256, 256).to(def_device)
im.requires_grad = True  # Tell PyTorch to track gradients for this tensor

# Step 2: Set up the optimizer
# Note: We pass [im] (the image) as the parameters to optimize
opt = torch.optim.Adam([im], lr=5e-2)

# Step 3: Define loss functions (same as before)
style_loss = StyleLossToTarget(style_im)
content_loss = ContentLossToTarget(content_im, target_layers=[6, 18, 25])

def combined_loss(x):
    return style_loss(x) * 0.2 + content_loss(x)

# Step 4: Manual optimization loop
for i in range(300):
    # Forward pass: compute loss
    loss = combined_loss(im)
    
    # Backward pass: compute gradients
    loss.backward()
    
    # Update the image using gradients
    opt.step()
    
    # Clear gradients for next iteration
    opt.zero_grad()

# Step 5: Display the result
show_image(im.clip(0, 1));

![](img_24.png)

**What does the code above do?**

This is the "manual" way to do the same thing:

1. **Create optimizable image**: `im.requires_grad = True`
2. **Create optimizer**: Pass the image as the parameter
3. **Training loop**:
   - `loss = combined_loss(im)`: Compute loss
   - `loss.backward()`: Compute gradients
   - `opt.step()`: Update image
   - `opt.zero_grad()`: Clear gradients

**Comparison with miniai:**

| Aspect | Raw PyTorch | miniai |
|--------|-------------|--------|
| Boilerplate | Write training loop manually | Automatic |
| Progress display | Need to add manually | Built-in callbacks |
| Logging | Need to add manually | Built-in callbacks |
| GPU handling | Manual `.to(device)` | DeviceCB handles it |
| Flexibility | Full control | Customize via callbacks |

**Pros and cons of each approach:**

**Raw PyTorch:**
- Pro: Full control, easy to understand what's happening
- Con: Lots of boilerplate code
- Con: Have to add progress tracking, logging manually

**miniai framework:**
- Pro: Less code, built-in features
- Pro: Easy to add functionality via callbacks
- Con: Abstraction hides some details

For production code or experiments, frameworks like miniai (or fastai, PyTorch Lightning) save time. For learning, understanding the raw approach is valuable!

## Perceptual Loss

Perceptual loss is simply the idea of **comparing images using VGG features instead of raw pixels.**

---

## Pixel loss vs perceptual loss

```python
# Pixel loss — compares raw pixel values
F.mse_loss(generated_im, content_im)

# Perceptual loss — compares VGG features
F.mse_loss(VGG(generated_im), VGG(content_im))
```

Pixel loss asks: *"are these pixels numerically identical?"*

Perceptual loss asks: *"do these images look similar to a neural network?"*

---

## Why pixel loss is not enough

Two images can be very different pixel-by-pixel but look almost identical to a human — for example, shifting an image one pixel to the right makes pixel loss huge but the images are perceptually the same. Conversely, two images can have similar average pixel values but look completely different.

VGG was trained on millions of images to recognize objects, so its features capture what actually matters perceptually — edges, textures, shapes, objects — rather than just raw numbers.

---

## In the context of style transfer

Both `ContentLossToTarget` and `StyleLossToTarget` are forms of perceptual loss:

- `ContentLossToTarget` → perceptual loss on **raw VGG features** (preserves structure)
- `StyleLossToTarget` → perceptual loss on **Gram matrices of VGG features** (preserves texture)

The name "perceptual" comes from the fact that you're measuring similarity the way a **perceptual system** (a trained neural network) would, rather than mathematically comparing pixel values.

---

## Summary

Congratulations! You've learned how to perform Neural Style Transfer. Let's recap the key concepts:

---

**Key Concepts:**

1. **Image Optimization**: Instead of optimizing model weights, we optimize the image pixels themselves.

2. **Feature Extraction**: Pre-trained CNNs (like VGG16) learn hierarchical features:
   - Early layers: Edges, colors, textures
   - Late layers: Shapes, objects, semantic content

3. **Content Loss**: Compares high-level features (late layers) to preserve structure.
   ```
   Content Loss = MSE(features_input, features_target)
   ```

4. **Gram Matrix**: Captures style by measuring feature correlations, removing spatial information.
   ```
   Gram[i,j] = correlation between features i and j
   ```

5. **Style Loss**: Compares Gram matrices to transfer texture/patterns.
   ```
   Style Loss = MSE(gram_input, gram_target)
   ```

6. **Combined Loss**: Balances content and style.
   ```
   Total Loss = α × Content Loss + β × Style Loss
   ```

---

**Things to Experiment With:**

- Different style images (paintings, textures, patterns)
- Different content/style weight ratios
- Different VGG layers for content and style
- Different image sizes
- Starting from content vs. random noise

---

**Further Reading:**

- Original paper: "A Neural Algorithm of Artistic Style" (Gatys et al., 2015)
- Feature visualization: https://distill.pub/2017/feature-visualization/
- Fast style transfer (real-time): "Perceptual Losses for Real-Time Style Transfer" (Johnson et al., 2016)